**CSI 4106 Introduction to Artificial Intelligence**  
*Assignment 1: Deep-Sea Mission Data Preparation*  
*Due: October 5, 2026, at 11 PM*

# Identification

Name: Kevin Govier  
Student number: 300282040   
Report title: Deep-Sea Mission Data Preparation Report

# 1. Data and objective

The fictional **Abyssal Survey and Robotics Centre (ASRC)** operates
autonomous underwater vehicles for seabed mapping, environmental
monitoring, and deep-ocean surveys. Each row represents one mission. The
predictors are available before the main survey begins. The response,
`mission_interrupted`, is `1` when the vehicle returned before
completing its planned survey and `0` otherwise.

The data are fully synthetic and were created for teaching.

- [Assignment 1 data
  directory](https://github.com/turcotte/csi4106-f26/tree/main/assignments-data/a1)
- [asrc_missions_raw.csv](https://raw.githubusercontent.com/turcotte/csi4106-f26/main/assignments-data/a1/asrc_missions_raw.csv)

Consult the assignment handout for the variable descriptions and ASRC
operating context.

# 2. Data preparation and exploratory analysis

## 2.1 Load and audit the dataset

Load the supplied CSV data. The notebook must remain able to obtain the
source file from the supplied GitHub URL without a manual upload. Report
its shape, first five rows, initial data types, and number of distinct
values per column.

In [ ]:
# Imports for getting the data
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from io import BytesIO

# Get data from GitHub URL
url = "https://raw.githubusercontent.com/turcotte/csi4106-f26/main/assignments-data/a1/asrc_missions_raw.csv"
response = requests.get(url)
response.raise_for_status()

# Read the CSV data
df = pd.read_csv(BytesIO(response.content))

# Get shape
print("Shape:", df.shape)
# Display first 5 rows
display(df.head(5))

# Get the initial data type and number of distinct values for every column
display(pd.DataFrame({
    "Initial Data Type":  df.dtypes,
    "Number of Distinct Values": df.nunique()
}))

Shape: (6371, 17)


,mission_id,deployment_batch,telemetry_protocol,planned_depth_m,planned_depth_ft,descent_current_speed_m_s,water_temperature_c,sonar_noise_db,navigation_drift_m,battery_health_pct,thruster_vibration_mm_s,missions_since_service,vehicle_class,seabed_terrain,maintenance_grade,firmware_current,mission_interrupted
0,ASRC-M000001,ASRC-B0253,asrc_tlm_v4,1400.0,4593.2,0.55,3.3000000000000003,66.0,10.4,87.0,3.1,14,compact,ridge,good,1,0
1,ASRC-M000002,ASRC-B0648,asrc_tlm_v4,2700.0,8858.3,0.68,3.0,78.5,49.900000000000006,83.0,3.9,4,heavy_duty,vent_field,good,1,1
2,ASRC-M000003,ASRC-B0340,asrc_tlm_v4,2870.0,9416.0,0.15,3.6,78.7,5.800000000000001,92.0,5.2,10,survey,vent_field,fair,1,1
3,ASRC-M000004,ASRC-B0620,asrc_tlm_v4,2840.0,9317.6,0.39,2.5,77.30000000000001,54.6,89.0,3.9,9,survey,trench,good,1,0
4,ASRC-M000005,ASRC-B0332,asrc_tlm_v4,2430.0,7972.4,0.34,2.8000000000000003,77.80000000000001,8.700000000000001,89.0,3.4000000000000004,8,heavy_duty,vent_field,excellent,1,0


,Initial Data Type,Number of Distinct Values
mission_id,str,6371
deployment_batch,str,850
telemetry_protocol,str,2
planned_depth_m,float64,582
planned_depth_ft,float64,576
descent_current_speed_m_s,float64,173
water_temperature_c,str,131
sonar_noise_db,str,339
navigation_drift_m,str,709
battery_health_pct,str,33


**Interpretation:** The CSV data was successfully loaded from the supplied GitHub URL. The shape was printed, showing 6371 rows and 17 columns, along with the first five rows. The initial data types and number of distinct values were also successfully printed for each column.

## 2.2 Handle missing values and data types

Investigate empty fields, `NA`, and `?`; convert them to `NaN`. Convert
numeric-like columns to numeric types, with parsing failures becoming
`NaN`. Report missing counts and percentages. Do not impute.

In [151]:
# Convert missing-value representations to actual missing values (NaN)
df.replace("", np.nan, inplace=True)
df.replace("NA", np.nan, inplace=True)
df.replace("?", np.nan, inplace=True)

# Convert numeric-like columns to numeric types
numericColumns = ["planned_depth_m", "planned_depth_ft", "descent_current_speed_m_s", "water_temperature_c", "sonar_noise_db", "navigation_drift_m", "battery_health_pct", "thruster_vibration_mm_s", "missions_since_service", "firmware_current", "mission_interrupted"]
for column in numericColumns:
    df[column] = pd.to_numeric(df[column], errors='coerce')
display(df.dtypes)

# Report the number and percentage of missing values in each affected column
missingValues = pd.DataFrame({
    "Number of Missing Values": df.isna().sum(),
    "Percentage of Missing Values": df.isna().mean()*100
})
display(missingValues[missingValues["Number of Missing Values"] > 0])

mission_id                       str
deployment_batch                 str
telemetry_protocol               str
planned_depth_m              float64
planned_depth_ft             float64
descent_current_speed_m_s    float64
water_temperature_c          float64
sonar_noise_db               float64
navigation_drift_m           float64
battery_health_pct           float64
thruster_vibration_mm_s      float64
missions_since_service         int64
vehicle_class                    str
seabed_terrain                   str
maintenance_grade                str
firmware_current               int64
mission_interrupted            int64
dtype: object

,Number of Missing Values,Percentage of Missing Values
water_temperature_c,169,2.652645
sonar_noise_db,123,1.930623
navigation_drift_m,120,1.883535
battery_health_pct,147,2.307330
thruster_vibration_mm_s,126,1.977712
maintenance_grade,129,2.024800


**Findings and method:** Empty fields, "NA", and "?" were successfully converted into actual missing values (NAN) using df.replace(). Numeric-like columns were successfully converted to numeric types using pd.to_numeric() with errors="coerce" to set values that cannot be parsed as NaN. The new datatypes of each column were then displayed to show this was a success. Lastly, the number and percentage of missing values in each affected column were displayed, and missing values were found in six columns: water_temperature_c with 169 missing values (2.7%), sonar_noise_db with 123 missing values (1.9%), navigation_drift_m with 120 missing values (1.9%), battery_health_pct with 147 missing values (2.3%), thruster_vibration_mm_s with 126 missing values (2.0%), and maintenance_grade with 129 missing values (2.0%).

## 2.3 Clean categorical attributes

Inspect every predictor represented by repeated text labels for
inconsistent category representations. Evaluate identifiers and
administrative codes in Task 2.6 rather than standardizing them merely
because they are text. Describe your investigation and cleaning method,
consolidate inconsistent labels where necessary, and show frequencies or
values before and after cleaning. Identify and state any meaningful
ordering among category levels.

In [152]:
# Inspect predictors represented by repeated text labels for inconsistent category representations
predictors = ["vehicle_class", "seabed_terrain", "maintenance_grade"]
print(f"Frequencies and values before cleaning:")
for column in predictors:
    display(df[column].value_counts(dropna=False))

# Clean inconsistent category representations
for column in predictors:
    df[column] = df[column].str.strip().str.lower().str.replace(" ", "_").str.replace("-", "_")

print(f"Frequencies and values after cleaning:")
for column in predictors:
    display(df[column].value_counts(dropna=False))

# Preserve the meaning of the levels of maintenance_grade (its levels have a meaningful order)
df["maintenance_grade"] = pd.Categorical(df["maintenance_grade"], categories=["poor", "fair", "good", "excellent"], ordered=True)

Frequencies and values before cleaning:


vehicle_class
survey        3140
compact       1843
heavy_duty    1292
SURVEY          31
Survey          18
 compact        15
Compact         13
Heavy Duty      11
heavy-duty       8
Name: count, dtype: int64

seabed_terrain
plain         2310
ridge         1770
trench        1223
vent_field     972
 plain          20
Plain           17
Ridge           14
 trench         13
RIDGE           10
Trench           9
Vent Field       8
vent-field       5
Name: count, dtype: int64

maintenance_grade
good         2481
excellent    2090
fair         1159
poor          416
NaN           129
Good           24
EXCELLENT      23
Excellent      18
 good          17
Poor            6
FAIR            5
Fair            2
 poor           1
Name: count, dtype: int64

Frequencies and values after cleaning:


vehicle_class
survey        3189
compact       1871
heavy_duty    1311
Name: count, dtype: int64

seabed_terrain
plain         2347
ridge         1794
trench        1245
vent_field     985
Name: count, dtype: int64

maintenance_grade
good         2522
excellent    2131
fair         1166
poor          423
NaN           129
Name: count, dtype: int64

**Findings and method:** I determined that the only predictor classes are vehicle_class, seabed_terrain, and maintenance_grade and I used value_counts() to inspect each of these columns for inconsistent category representations. Other text columns function as identifiers, such as mission_id, or are administrative, such as deployment_batch and telemetry_protocol. As such, these columns will be evaluated in Task 6. To clean the data, whitespace was removed, all text was converted to lowercase, and dashes and spaces were replaced with underscores. This successfully removed all inconsistent category representations, which can be seen when comparing the frequencies and values before vs after cleaning. Lastly, I determined that the levels of maintenance_grade have a meaningful order, as 'poor' should be lower than 'fair', which should be lower than 'good', which shouldbe lower than 'excellent'. I preserved the meaning of these levels using pd.Categorical(), providing the categories in the correct order and setting ordered=True.

## 2.4 Apply validity rules

Use the variable meanings, units, and ASRC operating context to define
and justify validation rules. Report the affected attributes and counts,
replace invalid measurements with `NaN`, and retain unusual but possible
observations. Do not guess corrections, reconstruct values from another
column, or remove entire missions.

In [153]:
# Validation rules
rules = {
    "planned_depth_m": (df["planned_depth_m"].notna() & df["planned_depth_m"] <= 0), # depth cannot be zero or negative
    "planned_depth_ft": (df["planned_depth_ft"].notna() & df["planned_depth_ft"] <= 0), # depth cannot be zero or negative
    "descent_current_speed_m_s": (df["descent_current_speed_m_s"].notna() & df["descent_current_speed_m_s"] < 0), # speed cannot be negative
    "water_temperature_c": (df["water_temperature_c"].notna() & df["water_temperature_c"] < -2), # the lowest temperature water can be before freezing is around -2 degrees Celsius (ocean water)
    "sonar_noise_db": (df["sonar_noise_db"].notna() & df["sonar_noise_db"] < 0), # noise decibals cannot be negative
    "navigation_drift_m": (df["navigation_drift_m"].notna() & df["navigation_drift_m"] < 0), # drift distance cannot be negative
    "battery_health_pct": (df["battery_health_pct"].notna() & ~df["battery_health_pct"].between(0, 100)), # percentage must be between 0 and 100
    "thruster_vibration_mm_s": (df["thruster_vibration_mm_s"].notna() & df["thruster_vibration_mm_s"] < 0), # vibration cannot be negative
    "missions_since_service": (df["missions_since_service"].notna() & df["missions_since_service"] < 0), # number of missions cannot be negative
    "firmware_current": (df["firmware_current"].notna() & ~df["firmware_current"].isin([0, 1])), # boolean value must be 0 or 1
    "mission_interrupted": (df["mission_interrupted"].notna() & ~df["mission_interrupted"].isin([0, 1])) # boolean value must be 0 or 1
}

# Report each affected attribute and the number of invalid values
print("Invalid values:")
for column, invalid in rules.items():
    if invalid.any():
        print(f"{column}: {invalid.sum()} invalid values")

# Replace demonstrably invalid measurements with NaN
for column, invalid in rules.items():
    df.loc[invalid, column] = np.nan

# Check that all invalid measurements were replaced
replaced = True
for column, invalid in rules.items():
    if not df.loc[invalid, column].isna().all(): # an invalid value was not replaced with NaN
        replaced = False
if replaced:
    print("\nAll invalid values were replaced")

Invalid values:
battery_health_pct: 6 invalid values

All invalid values were replaced


**Findings and justification:** I established rules for what values are invalid for each attribute based on their meaning, units, and context. Depth cannot be zero or negative, speed, noise decibals, drift distance, vibration, number of missions all cannot be negative, water temperaure cannot be less than -2, as this would always be ice, battery percentage must be between 0 and 100, and the boolean values (firmware_current and mision_interrupted) must be either 0 or 1. In addition, NaN values are not considered invalid. Unusual possible values were retained for most attributes by omitting an upper value limit. This is because although it may be extremely rare or odd for these attributes to have very high values, it is not technically impossible. Only values that were physically or logically impossible were considered invalid. Invalid values were replaced with NaN.

## 2.5 Explore distributions and relationships

Before removing any attributes, create a compact summary table and
labelled histogram grid for the quantitative measurement and count
attributes. Exclude identifiers, administrative codes, categorical
labels, binary indicators, and the response, even if they are stored
numerically. At minimum, report the non-missing count, mean, standard
deviation, quartiles, minimum, maximum, and skewness.

In [ ]:
# Get quantitative attributes
quantitatives = ["planned_depth_m", "planned_depth_ft", "descent_current_speed_m_s", "water_temperature_c", "sonar_noise_db", "navigation_drift_m", "battery_health_pct", "thruster_vibration_mm_s", "missions_since_service"]
qdf = df[quantitatives]

# Summary table
summary = qdf.describe().T
summary["skewness"] = qdf.skew()
summary = summary[["count", "mean", "std", "25%", "50%", "75%", "min", "max", "skewness"]]
display(summary)

# Histogram grid

,count,mean,std,25%,50%,75%,min,max,skewness
planned_depth_m,6371.0,2022.343431,1601.667573,780.00,1640.00,2870.00,280.00,37800.00,4.826957
planned_depth_ft,6371.0,6563.240057,4599.603801,2559.10,5380.60,9416.00,918.60,20636.50,0.924343
descent_current_speed_m_s,6371.0,0.422164,0.288578,0.21,0.36,0.56,-1.21,2.06,1.187400
water_temperature_c,6202.0,4.060480,2.268177,2.40,3.60,5.70,-1.50,13.30,0.534969
sonar_noise_db,6248.0,72.019062,6.820898,66.70,72.20,77.10,54.60,95.00,0.048921
navigation_drift_m,6251.0,21.752936,15.810234,9.80,18.00,29.30,0.80,132.20,1.483263
battery_health_pct,6218.0,87.952396,4.271393,85.00,88.00,91.00,67.00,99.00,-0.239280
thruster_vibration_mm_s,6245.0,4.651305,3.138782,2.50,3.90,6.00,0.20,45.00,2.187250
missions_since_service,6371.0,12.605713,7.864687,7.00,11.00,17.00,0.00,55.00,1.108305


Choose three attributes with different shapes. Discuss centre, spread,
symmetry or skewness, modality, and valid extreme observations.

**Detailed distribution analysis:**

Create a Pearson correlation heat map for the quantitative attributes.
Do not add identifiers, administrative codes, categorical encodings,
binary indicators, or the response unless specifically justified.
Discuss the strongest duplicate and non-duplicate relationships and
whether they are plausible. Add one joint plot involving at least two
attributes, and explain what it reveals beyond the univariate
histograms.

In [155]:
# Your code

**Relationship analysis:**

## 2.6 Select attributes for the analysis-ready dataset

Using the preceding exploration and the meaning of each attribute,
investigate unique identifiers, high-cardinality administrative fields,
quasi-constant attributes (one value in at least 99% of observations),
and deterministic or near-deterministic duplicate measurements. List and
justify every removed attribute. Demonstrate a suspected deterministic
or alternate-unit duplicate with a direct numerical relationship, such
as a conversion error or residual; correlation alone is insufficient
evidence of duplication. Do not remove strongly correlated attributes
when they measure different concepts.

In [156]:
# Your code

**Removed attributes and evidence:**

## 2.7 Examine the response

Report counts and proportions for `mission_interrupted`, include a
labelled bar chart, and briefly characterize the class balance. Do not
train a model.

In [157]:
# Your code

**Interpretation:**

## 2.8 Export the cleaned data

Retain the attributes you selected for the analysis-ready dataset and
`mission_interrupted`. Keep the retained predictors in their original
dataset order and place the response last. Leave unavailable or invalid
predictor measurements as `NaN`. Save the table as
`asrc_missions_cleaned.csv` without an index column. Reload the saved
CSV and verify its filename, shape, column names and order, categorical
labels, numeric measurements, missing values, row order, response, and
absence of an extra index column.

In [158]:
# Your code

**Final validation:**

# References

List all external sources using the IEEE numbered reference style
described in the handout. Cite adapted code in a nearby comment or
Markdown cell. If you did not use external sources, state “No external
sources were used.” You remain responsible for verifying, understanding,
and explaining everything in your submission.

# AI Use Declaration

**1. Did you use a generative AI tool while completing this
assignment?** This includes assistance with code, debugging, analysis,
interpretation, writing, translation, or finding sources.

**Answer:** No

If you answered **No**, mark the confirmation below and omit questions 2
through 6 and the AI Interaction Transcript.

- [✔] I confirm that I did not use generative AI in preparing this
  submission.